# Fine-Tune a Small RCM LLM for Denials Doctor

This notebook fine-tunes a small language model (Phi-4, Llama 3.2 8B, or Gemma 2 9B) on the Denials Doctor RCM knowledge base. The output is a fine-tuned model that can answer RCM/EHR questions for clients via API.

## Prerequisites

1. **Google Colab** - This notebook is designed for Colab (free T4 GPU or better)
2. **Hugging Face account** (optional) - For pushing model to hub
3. **Time estimate**: ~2-4 hours on T4 GPU for Phi-4 14B with QLoRA

In [ ]:
# Install dependencies
!pip install -qU \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    datasets \
    huggingface_hub \
    torch

print("Dependencies installed")

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

## 1. Clone the Knowledge Base

First, clone the Denials Doctor knowledge base repo:

In [ ]:
# Clone the repo
!git clone https://github.com/sriraajj-lab/denials-doctor-knowledge-base-.git
%cd denials-doctor-knowledge-base-

# List the structure
!ls -la

## 2. Build Training Dataset

Combine all JSONL training files into a single dataset:

In [ ]:
import json
import os
from datasets import Dataset

# Load all agent training files
training_data = []
agents_dir = "agents"

for filename in os.listdir(agents_dir):
    if filename.endswith("-training.jsonl"):
        filepath = os.path.join(agents_dir, filename)
        with open(filepath, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    training_data.append(json.loads(line))

print(f"Loaded {len(training_data)} training examples")

# Format in standard chat format
formatted = []
for item in training_data:
    formatted.append({
        "messages": [
            {"role": "user", "content": item["instruction"]},
            {"role": "assistant", "content": item["output"]}
        ]
    })

# Create HuggingFace dataset
dataset = Dataset.from_list(formatted)
print(f"Dataset size: {len(dataset)}")
print(f"Sample:\n{dataset[0]['messages']}")

## 3. Choose Base Model

Pick one of the following models:

In [ ]:
# Choose your model
MODEL_NAME = "microsoft/Phi-4-mini-instruct"  # Small model, ~4B params
# MODEL_NAME = "microsoft/Phi-4"  # ~14B, needs more GPU memory
# MODEL_NAME = "meta-llama/Llama-3.2-8B-Instruct"  # Solid 8B model
# MODEL_NAME = "google/gemma-2-9b-it"  # Alternative 9B model

print(f"Using: {MODEL_NAME}")

# For gated models (Llama, Gemma), log in first
# from huggingface_hub import login
# login()

## 4. Load Model with QLoRA

Uses 4-bit quantization to fit larger models on limited GPU memory:

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print(f"Trainable params: {model.num_parameters(only_trainable=True):,}")
print(f"Total params: {model.num_parameters():,}")

## 5. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    # Format as: <|user|>\nquestion\n<|assistant|>\nanswer
    texts = []
    for msg_list in examples["messages"]:
        user = msg_list[0]["content"]
        assistant = msg_list[1]["content"]
        if "Phi" in MODEL_NAME:
            text = f"<|user|>\n{user}\n<|assistant|>\n{assistant}<|endoftext|>"
        elif "Llama" in MODEL_NAME:
            text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{assistant}<|eot_id|>"
        elif "gemma" in MODEL_NAME.lower():
            text = f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{assistant}<end_of_turn>"
        else:
            text = f"Question: {user}\n\nAnswer: {assistant}"
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=2048,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["messages"])
print(f"Tokenized dataset: {len(tokenized_dataset)} examples")

## 6. Train the Model

In [ ]:
from trl import SFTTrainer

# Split into train/test
train_test = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_test["train"]
test_dataset = train_test["test"]

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

training_args = TrainingArguments(
    output_dir="./denials-doctor-rcm-fine-tuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    max_seq_length=2048,
    dataset_text_field=None,
)

# Start training
print("Starting training...")
trainer.train()

## 7. Save and Test the Model

In [ ]:
# Save the adapter
model_path = "./denials-doctor-rcm-adapter"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to {model_path}")
print("\nFile size:")
!ls -lh {model_path}/

In [ ]:
# Quick test
def test_model(prompt):
    if "Phi" in MODEL_NAME:
        text = f"<|user|>\n{prompt}\n<|assistant|>\n"
    elif "Llama" in MODEL_NAME:
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    else:
        text = f"Question: {prompt}\n\nAnswer: "
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test questions
tests = [
    "How do I appeal a Medicare medical necessity denial?",
    "What is the difference between CPT modifier 59 and XS?",
    "How long do I have to file a Medicare claim?",
]

for q in tests:
    print(f"Q: {q}")
    print(f"A: {test_model(q)}")
    print("---")

## 8. Merge and Export for Ollama/vLLM

To use the fine-tuned model with Ollama or vLLM, merge LoRA weights:

In [ ]:
from peft import PeftModel

# Reload base model without quantization for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Load and merge LoRA adapter
peft_model = PeftModel.from_pretrained(base_model, model_path)
merged_model = peft_model.merge_and_unload()

# Save merged model
merged_path = "./denials-doctor-rcm-merged"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print(f"Merged model saved to {merged_path}")
print("\nYou can now serve it with:")
print(f"  ollama create denials-doctor-rcm -f Modelfile")
print(f"  # Or with vLLM:")
print(f"  python -m vllm.entrypoints.openai.api_server --model {merged_path}")

## 9. (Optional) Push to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# model.push_to_hub("your-username/denials-doctor-rcm")
# tokenizer.push_to_hub("your-username/denials-doctor-rcm")

# print("Model pushed to Hugging Face Hub")